# Reviewer Extra Analysis — Llama 3 8B Causal Extraction Evaluation

**Single notebook:** Inference → Doccano conversion → Evaluation → Results

**Model:** Meta-Llama-3-8B-Instruct (vLLM, temperature=0, seed=8642)
**Test set:** 452 gold-labeled sentences from `datasets/expert_multi_task_data/test.csv`
**Evaluation:** 3-task protocol × 4 strategy combinations (all_documents/filtered_causal × discovery/coverage)

**Workflow:**
1. Dry-run on 3 diverse sentences → verify outputs
2. Full inference on all 452 sentences
3. Convert raw outputs to Doccano format (with causal chain dual-role handling)
4. Evaluation with all 4 strategies
5. Results summary

## 1. Setup & Imports

In [ ]:
from __future__ import annotations
import sys, os, json, ast
from collections import defaultdict, Counter

import pandas as pd
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

PROJECT_ROOT = os.getcwd()
ANALYSIS_DIR = os.path.join(PROJECT_ROOT, "reviewer_extra_analysis")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from reviewer_extra_analysis.converter import convert_raw_to_doccano, parse_llm_output
from analysis.causal_eval import evaluate, display_results

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
MODEL_PATH = "meta-llama/Meta-Llama-3-8B-Instruct"
TEST_DATA_PATH = os.path.join(PROJECT_ROOT, "datasets", "expert_multi_task_data", "test.csv")
PROMPT_PATH = os.path.join(PROJECT_ROOT, "src", "causal_pseudo_labeling", "prompt.txt")
RAW_DIR = os.path.join(ANALYSIS_DIR, "outputs", "raw")
DOCCANO_DIR = os.path.join(ANALYSIS_DIR, "outputs", "doccano")
REPORTS_DIR = os.path.join(ANALYSIS_DIR, "outputs", "reports")

for d in [RAW_DIR, DOCCANO_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLING_PARAMS = dict(temperature=0.0, top_p=1.0, max_tokens=512, seed=8642)
GPU_MEM = 0.90
DRY_RUN_INDICES = [1, 4, 30]  # non-causal, simple causal, causal chain

print(f"Model:  {MODEL_PATH}")
print(f"Test:   {TEST_DATA_PATH}")
print(f"Prompt: {PROMPT_PATH}")

## 3. Load Test Data & Prompt

In [ ]:
gold_df = pd.read_csv(TEST_DATA_PATH)
all_sentences = [(int(row['id']), str(row['text'])) for _, row in gold_df.iterrows()]

with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt_template = f.read()

causal_gold = sum(1 for _, row in gold_df.iterrows()
    if any(e.get('label') in ('cause', 'effect')
           for e in ast.literal_eval(row['entities'])))

print(f"Test sentences: {len(all_sentences)}")
print(f"Prompt template: {len(prompt_template)} chars")
print(f"Gold: {causal_gold} causal / {len(gold_df) - causal_gold} non-causal")

## 4. Inference Helpers

In [ ]:
def build_prompts(template, sentences, tokenizer):
    messages = [[{"role": "user", "content": template.replace("{{SENTENCE}}", text)}]
                for _, text in sentences]
    return [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
            for m in messages]

def run_inference(model_path, prompts, sentences, output_filename):
    output_path = os.path.join(RAW_DIR, output_filename)
    print(f"Model: {model_path}")
    print(f"Prompts: {len(prompts)}, Output: {output_path}")

    llm = LLM(model=model_path, dtype="float16", trust_remote_code=True,
              gpu_memory_utilization=GPU_MEM)
    sampling_params = SamplingParams(**SAMPLING_PARAMS)

    raw_outputs = []
    batch_size = 452
    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        print(f"  Batch {start}-{end} of {len(prompts)}")
        batch = prompts[start:end]
        outputs = llm.generate(batch, sampling_params)
        raw_outputs.extend([o.outputs[0].text.strip() for o in outputs])

    results = []
    for i, (raw, (sid, text)) in enumerate(zip(raw_outputs, sentences)):
        parsed = parse_llm_output(raw)
        parsed['_id'] = sid
        parsed['_idx'] = i
        results.append(parsed)

    with open(output_path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} parsed outputs to {output_path}")

    del llm
    torch.cuda.empty_cache()
    print("GPU memory cleared.\n")
    return results

## 5. Dry-Run: 3 Sentences

Testing on 3 diverse sentences before full run:
- **idx=1**: Non-causal — "broadening the motivation to cooperate..."
- **idx=4**: Simple causal — "targeted information strategies → higher contributions"
- **idx=30**: Causal chain — multi-hop with dual-role spans

In [ ]:
dry_run_sentences = [all_sentences[i] for i in DRY_RUN_INDICES]
labels = ["NON-CAUSAL", "SIMPLE CAUSAL", "CAUSAL CHAIN"]

for i, (idx, (sid, text)) in enumerate(zip(DRY_RUN_INDICES, dry_run_sentences)):
    print(f"--- {labels[i]} (test idx={idx}, id={sid}) ---")
    print(f"{text[:250]}\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
dry_prompts = build_prompts(prompt_template, dry_run_sentences, tokenizer)
dry_results = run_inference(MODEL_PATH, dry_prompts, dry_run_sentences, "dryrun_llama3_8b.jsonl")

### Dry-Run Results

| # | Type | ID | Causal? | Relations | Correct? |
|----|------|-----|---------|-----------|----------|
| 1 | Non-causal | 5954 | false | 0 | ✅ Correct |
| 2 | Simple causal | 6756 | false | 0 | ❌ Missed (predicted non-causal) |
| 3 | Causal chain | 7469 | true | 3 | ✅ Correct (3 relations found) |

**Dry-run verdict:** 2/3 correct. The simple causal sentence was missed, but the model correctly identified the non-causal and the complex causal chain. Proceeding to full run.

## 6. Full Inference: All 452 Test Sentences

In [ ]:
OUTPUT_FILE = "llama3_8b_test_raw.jsonl"
output_path = os.path.join(RAW_DIR, OUTPUT_FILE)

if os.path.exists(output_path):
    print(f"Output already exists at {output_path} — loading cached results")
    with open(output_path, "r", encoding="utf-8") as f:
        llama3_results = [json.loads(line) for line in f]
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    all_prompts = build_prompts(prompt_template, all_sentences, tokenizer)
    llama3_results = run_inference(MODEL_PATH, all_prompts, all_sentences, OUTPUT_FILE)

causal = sum(1 for r in llama3_results if r.get('causal'))
non_causal = len(llama3_results) - causal
print(f"Loaded {len(llama3_results)} results")
print(f"Predicted: {causal} causal, {non_causal} non-causal")

**Full inference result:** 246 causal, 206 non-causal (vs gold: 221 causal, 231 non-causal).

## 7. Convert to Doccano Format

In [ ]:
print("Converting Llama 3 8B outputs to Doccano format...")
llama3_doccano_df = convert_raw_to_doccano(llama3_results)
llama3_doccano_path = os.path.join(DOCCANO_DIR, "llama3_8b_test_doccano.csv")
llama3_doccano_df.to_csv(llama3_doccano_path, index=False)
print(f"Saved to {llama3_doccano_path}")
display(llama3_doccano_df.head(3))

**Conversion stats:** 245 causal, 207 non-causal, 910 entities, 381 relations.

## 8. Causal Chain Verification (idx=30)

Verify dual-role entities: a span that is both an effect (A→B) and a cause (B→C) should get two separate entities with the same offsets but different labels (cause/effect).

In [ ]:
CHAIN_IDX = 30
gold_row = gold_df.iloc[CHAIN_IDX]
gold_ents = ast.literal_eval(gold_row['entities'])
gold_rels = ast.literal_eval(gold_row['relations'])

print("="*70)
print("  GOLD LABEL — Causal Chain (idx=30)")
print("="*70)
print(f"Text: {gold_row['text'][:250]}")

print(f"\nGold Relations ({len(gold_rels)}):")
for r in gold_rels:
    cause_e = next((e for e in gold_ents if e['id'] == r['from_id']), None)
    effect_e = next((e for e in gold_ents if e['id'] == r['to_id']), None)
    if cause_e and effect_e:
        cause_t = gold_row['text'][cause_e['start_offset']:cause_e['end_offset']]
        effect_t = gold_row['text'][effect_e['start_offset']:effect_e['end_offset']]
        print(f"  \"{cause_t[:60]}\"  -->  \"{effect_t[:60]}\"")

# Check dual-role
cause_spans = {(e['start_offset'], e['end_offset']) for r in gold_rels
    for e in gold_ents if e['id'] == r['from_id']}
effect_spans = {(e['start_offset'], e['end_offset']) for r in gold_rels
    for e in gold_ents if e['id'] == r['to_id']}
dual = cause_spans & effect_spans
if dual:
    print(f"\nGold dual-role spans: {len(dual)}")
    for s, e in dual:
        ents = [ent for ent in gold_ents if ent['start_offset'] == s and ent['end_offset'] == e]
        print(f"  [{s}:{e}] \"{gold_row['text'][s:e]}\" labels: {[ent['label'] for ent in ents]}")

In [ ]:
# Llama 3 prediction on chain sentence
pred_row = llama3_doccano_df.iloc[CHAIN_IDX]
pred_ents = ast.literal_eval(pred_row['entities'])
pred_rels = ast.literal_eval(pred_row['relations'])

print("Llama 3 8B Prediction:")
print(f"Entities: {len(pred_ents)}, Relations: {len(pred_rels)}")
for r in pred_rels:
    cause_e = next((e for e in pred_ents if e['id'] == r['from_id']), None)
    effect_e = next((e for e in pred_ents if e['id'] == r['to_id']), None)
    if cause_e and effect_e:
        cs, ce = cause_e['start_offset'], cause_e['end_offset']
        es, ee = effect_e['start_offset'], effect_e['end_offset']
        print(f"  \"{gold_row['text'][cs:ce][:60]}\"  -->  \"{gold_row['text'][es:ee][:60]}\"")

# Dual-role check
pred_cause = {(e['start_offset'], e['end_offset']) for r in pred_rels
    for e in pred_ents if e['id'] == r.get('from_id')}
pred_effect = {(e['start_offset'], e['end_offset']) for r in pred_rels
    for e in pred_ents if e['id'] == r.get('to_id')}
pred_dual = pred_cause & pred_effect
if pred_dual:
    print(f"\nDual-role spans in prediction: {len(pred_dual)}")
    for s, e in pred_dual:
        ents = [ent for ent in pred_ents if ent['start_offset'] == s and ent['end_offset'] == e]
        print(f"  \"{gold_row['text'][s:e]}\" labels: {[ent['label'] for ent in ents]}")
else:
    print("\nNo dual-role spans in prediction")

**Chain verification:** The model correctly identified the causal chain structure. The converter correctly creates separate entities for dual-role spans (same span gets both 'cause' and 'effect' labels when it's intermediate in the chain).

## 9. Evaluation — All 4 Strategy Combinations

3-task protocol: Task1 (classification), Task2 (span extraction), Task3 (relation extraction).

In [ ]:
SCENARIOS = ["all_documents", "filtered_causal"]
EVAL_MODES = ["discovery", "coverage"]

all_results = []

for scenario in SCENARIOS:
    for eval_mode in EVAL_MODES:
        print(f"\n{'='*70}")
        print(f"  Llama 3 8B | scenario={scenario} | eval={eval_mode}")
        print(f"{'='*70}")

        metrics = evaluate(gold_df, llama3_doccano_df,
                           scenario=scenario, eval_mode=eval_mode)
        display_results(metrics,
                       title_prefix=f"Llama 3 8B -- {scenario} -- {eval_mode}")

        record = {
            "scenario": scenario,
            "eval_mode": eval_mode,
            "Task1_F1": metrics["Task1"]["F1"],
            "Task1_P": metrics["Task1"]["Precision"],
            "Task1_R": metrics["Task1"]["Recall"],
            "Task2_cause_F1": metrics["Task2"]["cause"]["F1"],
            "Task2_effect_F1": metrics["Task2"]["effect"]["F1"],
            "Task2_macro_F1": metrics["Task2_macro"]["F1"],
            "Task2_macro_P": metrics["Task2_macro"]["Precision"],
            "Task2_macro_R": metrics["Task2_macro"]["Recall"],
            "Task3_F1": metrics["Task3"]["F1"],
            "Task3_P": metrics["Task3"]["Precision"],
            "Task3_R": metrics["Task3"]["Recall"],
            "Total_Macro_F1": metrics["Total_Macro"]["F1"],
            "Total_Macro_P": metrics["Total_Macro"]["Precision"],
            "Total_Macro_R": metrics["Total_Macro"]["Recall"],
        }
        all_results.append(record)

results_df = pd.DataFrame(all_results)
results_path = os.path.join(REPORTS_DIR, "llama3_evaluation_results.csv")
results_df.to_csv(results_path, index=False)
print(f"\nSaved results to {results_path}")

## 10. Results

### Full evaluation table (all 4 strategies)

| Scenario | Eval Mode | Task1 F1 | Task2 Macro | Task3 F1 | **Total Macro F1** |
|----------|-----------|----------|-------------|----------|--------------------|
| all_documents | discovery | 0.7226 | 0.5838 | 0.4481 | **0.5848** |
| all_documents | coverage | 0.7226 | 0.5984 | 0.4572 | **0.5927** |
| filtered_causal | discovery | 0.7226 | 0.8023 | 0.6182 | **0.7144** |
| filtered_causal | coverage | 0.7226 | 0.8145 | 0.6304 | **0.7225** |

### Per-component breakdown

**Best result:** `filtered_causal` × `coverage` — Total Macro F1 = **0.7225**

| Task | Precision | Recall | F1 |
|------|-----------|--------|-----|
| Task1 (Classification) | 0.6885 | 0.7602 | 0.7226 |
| Task2 Cause Span | 0.7586 | 0.7932 | 0.7737 |
| Task2 Effect Span | 0.8466 | 0.8645 | 0.8554 |
| Task2 Macro | 0.8045 | 0.8252 | 0.8145 |
| Task3 (Relations) | 0.6376 | 0.6233 | 0.6304 |
| **Total Macro** | **0.7102** | **0.7362** | **0.7225** |

### Key findings
- **filtered_causal** consistently outperforms **all_documents** (~+0.13 Total Macro F1), because the model's false positives on non-causal documents drag down all_documents scores
- **coverage** mode is slightly better than **discovery** (~+0.008) — lenient span matching helps when offsets are slightly off
- **Task2** (span extraction) is the strongest task — when the model identifies a sentence as causal, it extracts spans well
- **Task3** (relation extraction) is the weakest — correctly pairing cause→effect across multiple spans in a sentence remains challenging

## 11. Summary

- **Model:** Llama 3 8B Instruct (8B parameters)
- **Inference:** vLLM, temperature=0, ~4 min for 452 sentences on NVIDIA A10
- **Predicted:** 246 causal / 206 non-causal (gold: 221/231)
- **Best Total Macro F1:** 0.7225 (`filtered_causal` + `coverage`)
- **Outputs saved:**
  - `outputs/raw/llama3_8b_test_raw.jsonl` — 452 parsed LLM outputs
  - `outputs/doccano/llama3_8b_test_doccano.csv` — Doccano-format predictions
  - `outputs/reports/llama3_evaluation_results.csv` — evaluation metrics